# Where does esports prize money go? Country-level analysis, 1998–2024

Part 2 of the esports prize money analysis: how prize money is distributed across countries, and how this changed over time.

## Research questions
1. How concentrated is prize money across countries?
2. How did the set of leading countries change?
3. Which countries earn the most per player?

## Data
Kaggle dataset `jeanhousseau/esports-earnings-history` (data from Esports Earnings): yearly rankings of countries, players (top 100 per year), games and teams, 1998–2024.

## 1. Loading the data
Load the country and player files and check their size and the covered period.

In [ ]:
import pandas as pd

country = pd.read_csv("esports_country_earnings_by_year.csv")
player = pd.read_csv("esports_player_earnings_by_year.csv")

print(country.shape, player.shape)
print(country["year"].min(), country["year"].max())
print("Countries:", country["country_name"].nunique())
print(player.groupby("year").size().tail(10))   # number of players in each yearly ranking
country.head()

The country file has 2,122 country-year records for 178 countries (1998–2024). The player file lists only the top 100 players per year, so it is not used to measure concentration among players.

## 2. Sanity check: totals by country vs. by game
Yearly prize money summed over countries should be close to the yearly total summed over games.

In [ ]:
game = pd.read_csv("esports_game_earnings_by_year.csv")

check = pd.DataFrame({
    "by_country": country.groupby("year")["total_earnings"].sum(),
    "by_game": game.groupby("year")["total_earnings"].sum(),
})
check["ratio"] = (check["by_country"] / check["by_game"]).round(3)
print(check.loc[2015:])

Country totals cover about 90–98% of the totals by game (lowest in 2019, 0.896), so some prize money is not attributed to a country.

## 3. Concentration across countries
For each year: the number of countries with prize money, the share of the 5 largest countries, and the Herfindahl index (HHI, the sum of squared shares; closer to 1 means higher concentration).

In [ ]:
rows = []
for year, g in country.groupby("year"):
    sh = g["total_earnings"] / g["total_earnings"].sum()
    rows.append((year, len(g), sh.sort_values(ascending=False).head(5).sum(), (sh ** 2).sum()))
conc_c = pd.DataFrame(rows, columns=["Year", "Countries", "Top5Share", "HHI"]).set_index("Year")
print(conc_c.loc[2005:].round(3))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

plot_data = conc_c.loc[2005:]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))

plot_data["Top5Share"].plot(ax=ax[0], marker="o")
ax[0].set_title("Share of top-5 countries in yearly prize money")
ax[0].set_ylim(0, 1)

plot_data["Countries"].plot(ax=ax[1], marker="o")
ax[1].set_title("Number of countries with prize money")

for a in ax:
    a.xaxis.set_major_locator(MaxNLocator(integer=True))
    a.set_xlabel("Year")

plt.tight_layout()
plt.savefig("fig3_country_concentration.png", dpi=150)
plt.show()

The number of countries with prize money grew from 49 (2005) to 144 (2024). The top-5 share was about 0.63–0.71 in 2005–2014, fell from 0.60 (2016) to 0.48 (2017), and has been about 0.47–0.52 since. HHI fell from about 0.11–0.16 to about 0.064–0.080. This break is earlier than the 2020 break in the game-level data. Before 2005 there were very few countries, so shares are mechanically high and are not shown.

## 4. Leading countries by decade
The five largest countries in each decade and their share of that decade's prize money. The label 1990 covers 1998–1999 and the label 2020 covers 2020–2024 only.

In [ ]:
country["Decade"] = (country["year"] // 10) * 10
dec = country.groupby(["Decade", "country_name"])["total_earnings"].sum().reset_index()
for d, g in dec.groupby("Decade"):
    total = g["total_earnings"].sum()
    top5 = g.sort_values("total_earnings", ascending=False).head(5)
    print(d, dict(zip(top5["country_name"], (top5["total_earnings"] / total).round(2).tolist())))

The leading countries changed from the United States and Korea (2000s) to China and the United States (2010s), and to China, the United States, Russia, Korea and Brazil in 2020–2024.

## 5. Prize money per player-year by country, 2015–2024
Total prize money divided by the number of player-years in the yearly country rankings, for countries with at least 100 player-years. This is a rough measure: it includes players with very small payouts, counts a player in every year he or she earned prize money, and is not adjusted for inflation.

In [ ]:
recent = country[country["year"] >= 2015]
pc = recent.groupby("country_name").agg(
    earnings=("total_earnings", "sum"),
    player_years=("number_of_players", "sum"),
)
pc = pc[pc["player_years"] >= 100]
pc["per_player_year"] = (pc["earnings"] / pc["player_years"]).round(0)
print(pc.sort_values("earnings", ascending=False).head(15))

China leads both in total prize money and per player-year. The United States earns almost as much in total, but spread over many more player-years. Smaller countries such as Denmark and Ukraine earn more per player-year.

## Limitations
- Country is the nationality of the player, not the location of the tournament.
- Country totals cover about 90–98% of the yearly totals by game.
- The player ranking file lists only the top 100 players per year, so concentration among players cannot be measured.
- 2024 covers only part of the year and may be incomplete.
- Prize money is in nominal USD, without inflation adjustment, and excludes salaries, sponsorship and streaming income.
- The analysis is descriptive; explanations for the changes are hypotheses and are not tested statistically.